# מחולל חוברות חברה מבוססות מקורות
מטלה 1 · הרחבה של השבוע הראשון, יום 5

המחברת מציגה את ההפרדה בין בחירת דפים, שמירת ראיות וכתיבה לקהלים שונים. בתיק ההגשה הנוכחי נכללו בדיקות מקומיות ותיעוד של ניסיון שליפה. טרם התקבלו חוברות חברה אמיתיות. יש לבחור את תיקיית הפרויקט כתיקיית העבודה של הליבה.

In [1]:
from pathlib import Path
from config import Settings, client_for
from storage import read_json
from context_manager import build_context
from workflow import run_workflow, saved_links_step, replay, output_path
from evaluation import MAIN, read_text
from llm_service import aggregate_stream

MODE = "inspect"  # "live" runs the model, "replay" reads an existing run
RUN_DIRECTORY = None  # Set to a validated A_links run to continue live, or a completed run to replay.
settings = Settings.load()
for company in settings.companies:
    print(company.id, company.name, company.url)
print("Source character budget:", settings.budget)

A Plausible Analytics https://plausible.io/
B Fathom Analytics https://usefathom.com/
Source character budget: 12000


## שליפת אתרים
בשלב האיסוף מבקשים פעם אחת את דף הבית ומחלצים ממנו טקסט וקישורים. פייתון מנרמל כתובות ושומר רשימת מועמדים לפני הבחירה. תשובה פסולה עוצרת את שליפת כל הקישורים שנבחרו. אפשר לעיין כאן בתוצאות בדיקת הגישה ששמורות בחבילה.

In [2]:
files = sorted(Path("runs/preflight").glob("*/sites.json"))
if files:
    latest = files[-1]
    for site in read_json(latest):
        print(site["company"], site["status"], site["chars"], site["error"])
    check = latest.with_name("connectivity.json")
    if check.exists():
        for entry in read_json(check)["sockets"]:
            print(entry["host"], entry.get("tcp", entry.get("error")), entry.get("winerror", ""))
        print(read_json(check)["provider"])


Plausible Analytics failed 0 ConnectionError
Fathom Analytics failed 0 ConnectionError
plausible.io PermissionError 10013
usefathom.com PermissionError 10013
127.0.0.1 ConnectionRefusedError 10061
{'base_url': 'http://127.0.0.1:11434/v1', 'model': 'qwen3:8b', 'checked': True, 'available': False, 'error': 'ConnectionError'}


## בדיקת תקציב באמצעות טקסט מדומה
בדוגמה הקצרה דף הבית ארוך, לדף השני יש שני תווים בלבד ולחברה השנייה יש מקור ארוך נוסף. נבחן כמה תווים קיבל כל מקור מתוך תקציב משותף של 22 תווים.

In [3]:
def fixture(sid, text):
    return dict(source_id=sid, company_id=sid[0], title="בדיקה", url="https://fixture.invalid/", status="ok", text=text)

sample = build_context([fixture("A1", "א"*1200), fixture("A2", "אב"), fixture("B1", "ב"*1200)], 22)
for record in sample["decisions"]:
    print(record["source_id"], record["state"], record["supplied_chars"], record["reason"])

A1 truncated 10 shared character budget
A2 included 2 full source
B1 truncated 10 shared character budget


## בדיקת זרימה מקומית
אירועים ללא טקסט אינם משנים את התצוגה. בריצה חיה רק החוברת למועמדי חברה ב׳ משתמשת בזרימה. תא ההרצה שלהלן מעדכן את אותו אזור תצוגה עם כל תוספת.

In [4]:
from types import SimpleNamespace as Event

def chunk(text=None, finish=None):
    return Event(choices=[Event(delta=Event(content=text), finish_reason=finish)])

events = [chunk("טקסט "), chunk(None), Event(choices=[]), chunk("בדיקה"), chunk(finish="stop")]
result, error, usage = aggregate_stream(events, print)
assert result == "טקסט בדיקה" and error is None

טקסט 
טקסט בדיקה


## הפקת ערכת המידע או שחזור
יש לערוך את התצורה בתא הראשון. במצב הפקה דרוש מפתח למסלול שנבחר. במצב שחזור יש להגדיר נתיב להרצה שהסתיימה. פלטים שנפסלו נשמרים בתיקיית התגובות המקוריות. בחירת דפים, כתיבת חוברות, השוואה וגרסת בסיס צורכות שמונה קריאות בהפקה רגילה.

ברירת המחדל היא חיבור מקומי ל־Ollama. יש לבחור תג דגם מותקן מתוך הפלט של ollama list. בבדיקה המקומית שני האתרים נחסמו ברמת החיבור, ושירות המודל לא ענה. בדיקת הקוד בוצעה באמצעות נתונים מדומים בלבד.

כאשר יש מקור חסר או אוסף קצר, מקום פנוי מאפשר בחירה מנומקת נוספת מתוך קישורים שטרם נשלפו. אם תוצר כתיבה שהושלם נכשל באימות, מותר ניסיון נוסף אחד באותן הוראות ובאותו הקשר. שתי התגובות נשמרות. חוברת מתקבלת לפי הניסיון שעבר את בדיקת התוכן המבנית.

In [5]:
if MODE == "live":
    from IPython.display import Markdown, display
    handle = None
    def progressive(content):
        global handle
        if handle is None:
            handle = display(Markdown(content), display_id=True)
        else:
            handle.update(Markdown(content))
    display_options = dict(show=lambda content: display(Markdown(content)), update=progressive)
    if RUN_DIRECTORY is None:
        RUN_DIRECTORY = run_workflow(settings, client_for(settings), **display_options)
    else:
        RUN_DIRECTORY = saved_links_step(RUN_DIRECTORY, settings, continue_run=True, **display_options)
elif MODE == "replay":
    if RUN_DIRECTORY is None:
        raise ValueError("Set RUN_DIRECTORY to a saved generation run.")
elif MODE != "inspect":
    raise ValueError("Choose inspect, replay or live.")

In [6]:
if RUN_DIRECTORY is not None:
    from IPython.display import Markdown, display
    root = Path(RUN_DIRECTORY)
    summary = replay(root)
    for cid in "AB":
        print(cid, read_json(root / "data" / f"{cid}_selection.json")["links"])
    for name in MAIN + ["comparison", "baseline"]:
        print(name, summary[name])
        if MODE != "live":
            display(Markdown(read_text(output_path(root, name))))
else:
    print("No live brochure set is attached to this notebook.")

No live brochure set is attached to this notebook.


## ביקורת שמונה טענות וניסוי הנחיות
כלל הדגימה הוא שתי הטענות העובדתיות המהותיות הראשונות בכל חוברת לפי סדר הקריאה, לרבות טענה ללא הפניה. בתיקיית review מופיעים גוף כל חוברת וההקשר המדויק שלה. יש להעתיק את הטענה והראיה, לבחור הכרעה ולנמק אותה. במקרה של חסר יש לתעד אותו.

בניסוי משווים לחוברת הלקוחות של חברה א׳ גרסת בסיס עם אותן דרישות אורך, שפה והפניות. בדקו נאמנות, התאמה לקהל, כיסוי ובהירות. לכל ממד בחרו דוגמה מכל פלט וכתבו ממצא. לאחר מילוי הטפסים, פקודת review בודקת התאמת ציטוטים ומייצאת את ההערכה. עיינו גם בעקביות בין חוברות אותה חברה ובמידת התמיכה בטענות ההשוואה.

In [7]:
if RUN_DIRECTORY is not None:
    import csv
    for name in ("claim_audit.csv", "prompt_experiment.md"):
        file = Path(RUN_DIRECTORY) / "artifacts" / name
        if file.exists():
            if file.suffix == ".csv":
                with file.open(encoding="utf-8-sig", newline="") as stream:
                    for row in csv.DictReader(stream):
                        print(row)
            else:
                display(Markdown(read_text(file)))
        else:
            print(name, "awaiting completed evidence review")